<a href="https://colab.research.google.com/github/priyan1436ei-lab/neural-network--scratch/blob/main/neural_network_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Neural Network from Scratch (NumPy only)
Architecture: 2 inputs -> 4 hidden neurons (sigmoid) -> 1 output (sigmoid)
Task: learn XOR (a problem a single neuron CANNOT solve)
"""
import numpy as np

np.random.seed(42)

# ---------- 1. DATA ----------
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)   # (4, 2)
y = np.array([[0], [1], [1], [0]], dtype=float)               # (4, 1)

# ---------- 2. ACTIVATION ----------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(a):
    # 'a' is already sigmoid(z), so derivative = a * (1 - a)
    return a * (1 - a)

# ---------- 3. INITIALISE WEIGHTS ----------
input_size, hidden_size, output_size = 2, 4, 1
W1 = np.random.randn(input_size, hidden_size)    # (2, 4)
b1 = np.zeros((1, hidden_size))                  # (1, 4)
W2 = np.random.randn(hidden_size, output_size)   # (4, 1)
b2 = np.zeros((1, output_size))                  # (1, 1)

learning_rate = 0.5
epochs = 10000

# ---------- 4. TRAINING LOOP ----------
for epoch in range(epochs):
    # ---- Forward pass ----
    z1 = X @ W1 + b1          # hidden pre-activation
    a1 = sigmoid(z1)          # hidden activation
    z2 = a1 @ W2 + b2         # output pre-activation
    a2 = sigmoid(z2)          # prediction

    # ---- Loss (Mean Squared Error) ----
    loss = np.mean((y - a2) ** 2)

    # ---- Backward pass (chain rule) ----
    d_a2 = 2 * (a2 - y) / len(X)              # dLoss/da2
    d_z2 = d_a2 * sigmoid_derivative(a2)      # dLoss/dz2
    d_W2 = a1.T @ d_z2                        # dLoss/dW2
    d_b2 = np.sum(d_z2, axis=0, keepdims=True)

    d_a1 = d_z2 @ W2.T                        # push error back to hidden layer
    d_z1 = d_a1 * sigmoid_derivative(a1)
    d_W1 = X.T @ d_z1
    d_b1 = np.sum(d_z1, axis=0, keepdims=True)

    # ---- Gradient descent update ----
    W2 -= learning_rate * d_W2
    b2 -= learning_rate * d_b2
    W1 -= learning_rate * d_W1
    b1 -= learning_rate * d_b1

    if epoch % 1000 == 0:
        print(f"Epoch {epoch:5d} | Loss: {loss:.6f}")

# ---------- 5. PREDICT ----------
def predict(X):
    return sigmoid(sigmoid(X @ W1 + b1) @ W2 + b2)

print("\nFinal predictions:")
for inp, out in zip(X, predict(X)):
    print(f"{inp} -> {out[0]:.4f}  (rounded: {int(round(out[0]))})")

In [ ]:
import matplotlib.pyplot as plt

# Store the loss values during training
# For this, add a history list before your training loop:
# loss_history = []

# Inside your training loop, after calculating loss:
# loss_history.append(loss)

plt.plot(loss_history)
plt.title("Neural Network Training Loss")
plt.xlabel("Training Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Your existing training loss values
loss_history = [
    0.283190,
    0.183325,
    0.020997,
    0.005533,
    0.002786,
    0.001768,
    0.001262,
    0.000966,
    0.000774,
    0.000642
]

epochs = list(range(0, 10000, 1000))

plt.figure(figsize=(8, 5))
plt.plot(epochs, loss_history, marker="o")
plt.title("Neural Network Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

In [ ]:
def predict_xor(a, b):
    inputs = np.array([[a, b]], dtype=float)

    hidden = sigmoid(inputs @ W1 + b1)
    output = sigmoid(hidden @ W2 + b2)

    prediction = int(output[0, 0] >= 0.5)

    print("Input:", a, b)
    print("Probability:", round(float(output[0, 0]), 4))
    print("Predicted XOR:", prediction)


predict_xor(0, 0)
predict_xor(0, 1)
predict_xor(1, 0)
predict_xor(1, 1)

In [ ]:
import numpy as np

class NeuralNetwork:
    def __init__(self, layer_sizes, lr=0.1, seed=42):
        """layer_sizes example: [2, 8, 1] = 2 inputs, 8 hidden, 1 output"""
        rng = np.random.default_rng(seed)
        self.lr = lr
        self.W, self.b = [], []
        for i in range(len(layer_sizes) - 1):
            # He initialisation: works well with ReLU
            self.W.append(rng.standard_normal((layer_sizes[i], layer_sizes[i+1]))
                          * np.sqrt(2 / layer_sizes[i]))
            self.b.append(np.zeros((1, layer_sizes[i+1])))

    @staticmethod
    def relu(z):
        return np.maximum(0, z)

    @staticmethod
    def sigmoid(z):
        return 1 / (1 + np.exp(-z))

    def forward(self, X):
        self.A = [X]                         # store every layer's output
        for i in range(len(self.W)):
            z = self.A[-1] @ self.W[i] + self.b[i]
            last = (i == len(self.W) - 1)
            self.A.append(self.sigmoid(z) if last else self.relu(z))
        return self.A[-1]

    def backward(self, y):
        m = len(y)
        # With sigmoid output + cross-entropy loss, the gradient simplifies:
        dz = (self.A[-1] - y) / m
        for i in reversed(range(len(self.W))):
            dW = self.A[i].T @ dz
            db = dz.sum(axis=0, keepdims=True)
            if i > 0:                                  # push error back
                dz = (dz @ self.W[i].T) * (self.A[i] > 0)   # ReLU derivative
            self.W[i] -= self.lr * dW
            self.b[i] -= self.lr * db

    def loss(self, y):
        p = np.clip(self.A[-1], 1e-9, 1 - 1e-9)
        return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

    def train(self, X, y, epochs=5000, print_every=1000):
        history = []
        for e in range(epochs):
            self.forward(X)
            history.append(self.loss(y))
            self.backward(y)
            if e % print_every == 0:
                print(f"Epoch {e:5d} | Loss: {history[-1]:.6f}")
        return history

    def predict(self, X):
        return self.forward(X)


if __name__ == "__main__":
    X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
    y = np.array([[0],[1],[1],[0]], dtype=float)

    net = NeuralNetwork([2, 8, 1], lr=0.1)
    net.train(X, y, epochs=5000)
    print("\nPredictions:")
    print(net.predict(X).round(3))

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output
from google.colab import output

# Enable interactive widgets in Google Colab
output.enable_custom_widget_manager()

# Check that the trained model exists
required_vars = ["W1", "b1", "W2", "b2"]
missing = [name for name in required_vars if name not in globals()]

if missing:
    raise ValueError(
        f"Missing trained model variables: {missing}. "
        "Run your training cell first."
    )

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))


def draw_network(a, b):
    # Forward pass using your trained weights
    inputs = np.array([[a, b]], dtype=float)

    z1 = inputs @ W1 + b1
    hidden = sigmoid(z1)

    z2 = hidden @ W2 + b2
    probability = float(sigmoid(z2)[0, 0])
    prediction = int(probability >= 0.5)
    expected = int(a != b)

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.set_xlim(-0.7, 3.7)
    ax.set_ylim(-0.7, 4.7)
    ax.axis("off")

    # Node positions: 2 input, 4 hidden, 1 output
    input_y = [1.5, 3.5]
    hidden_y = [0.5, 1.5, 2.5, 3.5]
    output_y = [2.0]

    # Draw input -> hidden connections
    for i, y1 in enumerate(input_y):
        for j, y2 in enumerate(hidden_y):
            weight = W1[i, j]
            ax.plot(
                [0, 1.5], [y1, y2],
                color="green" if weight >= 0 else "red",
                linewidth=0.7 + 2.0 * min(abs(weight), 2),
                alpha=0.65,
                zorder=1
            )
            ax.text(
                0.65, (y1 + y2) / 2,
                f"{weight:.2f}",
                fontsize=7, color="black",
                bbox=dict(facecolor="white",
                          alpha=0.7, edgecolor="none", pad=1)
            )

    # Draw hidden -> output connections
    for j, y in enumerate(hidden_y):
        weight = W2[j, 0]
        ax.plot(
            [1.5, 3], [y, output_y[0]],
            color="green" if weight >= 0 else "red",
            linewidth=0.7 + 2.0 * min(abs(weight), 2),
            alpha=0.65,
            zorder=1
        )
        ax.text(
            2.25, (y + output_y[0]) / 2,
            f"{weight:.2f}",
            fontsize=8, color="black",
            bbox=dict(facecolor="white",
                      alpha=0.7, edgecolor="none", pad=1)
        )

    # Draw neurons and display their activation values
    for i, y in enumerate(input_y):
        circle = plt.Circle(
            (0, y), 0.22, color="#8ecae6",
            ec="black", zorder=3
        )
        ax.add_patch(circle)
        ax.text(0, y, str(int(inputs[0, i])),
                ha="center", va="center",
                fontsize=12, weight="bold", zorder=4)

    for j, y in enumerate(hidden_y):
        value = float(hidden[0, j])
        circle = plt.Circle(
            (1.5, y), 0.22,
            color=plt.cm.RdYlGn(value),
            ec="black", zorder=3
        )
        ax.add_patch(circle)
        ax.text(1.5, y, f"{value:.2f}",
                ha="center", va="center",
                fontsize=9, zorder=4)

    circle = plt.Circle(
        (3, output_y[0]), 0.28,
        color="#ffcc80", ec="black", zorder=3
    )
    ax.add_patch(circle)
    ax.text(3, output_y[0], f"{probability:.2f}",
            ha="center", va="center",
            fontsize=10, weight="bold", zorder=4)

    ax.text(0, 4.2, "INPUT", ha="center", weight="bold")
    ax.text(1.5, 4.2, "HIDDEN LAYER", ha="center", weight="bold")
    ax.text(3, 2.65, "OUTPUT", ha="center", weight="bold")

    ax.set_title(
        f"XOR Neural Network | Input=({a}, {b}) | "
        f"Prediction={prediction} | Expected={expected}",
        fontsize=12
    )

    plt.tight_layout()
    plt.show()

    print(f"Input: ({a}, {b})")
    print(f"Output probability: {probability:.4f}")
    print(f"Predicted XOR: {prediction}")
    print(f"Expected XOR: {expected}")
    print("Status:", "CORRECT" if prediction == expected else "INCORRECT")


# Interactive input controls
a_slider = widgets.IntSlider(
    value=0, min=0, max=1, step=1, description="Input A"
)

b_slider = widgets.IntSlider(
    value=0, min=0, max=1, step=1, description="Input B"
)

ui = widgets.VBox([a_slider, b_slider])

out = widgets.interactive_output(
    draw_network,
    {"a": a_slider, "b": b_slider}
)

display(ui, out)


ValueError: Missing trained model variables: ['W1', 'b1', 'W2', 'b2']. Run your training cell first.

In [ ]:
"""
Professional-style neural network in PyTorch (CNN on MNIST)
Includes: GPU support, data augmentation, BatchNorm, Dropout, AdamW,
OneCycle learning-rate schedule, label smoothing, best-model checkpointing.
"""
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ---------- 1. CONFIG ----------
SEED, EPOCHS, BATCH, MAX_LR = 42, 10, 128, 2e-3
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ---------- 2. DATA ----------
MEAN, STD = 0.1307, 0.3081          # MNIST pixel statistics
train_tf = transforms.Compose([
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1)),   # augmentation
    transforms.ToTensor(),
    transforms.Normalize((MEAN,), (STD,)),
])
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MEAN,), (STD,)),
])

train_ds = datasets.MNIST("data", train=True, download=True, transform=train_tf)
test_ds = datasets.MNIST("data", train=False, download=True, transform=test_tf)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=2, pin_memory=(device == "cuda"))
test_loader = DataLoader(test_ds, batch_size=512, num_workers=2)

# ---------- 3. MODEL ----------
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),                                   # 28 -> 14
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),                                   # 14 -> 7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10),          # raw scores (logits); softmax is inside the loss
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = CNN().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# ---------- 4. LOSS, OPTIMIZER, SCHEDULER ----------
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=MAX_LR, epochs=EPOCHS, steps_per_epoch=len(train_loader))

# ---------- 5. TRAIN / EVALUATE ----------
def evaluate(loader):
    model.eval()                                   # turns off dropout, fixes BatchNorm
    correct = total = 0
    with torch.no_grad():                          # no gradients needed for testing
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            correct += (model(images).argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return correct / total

best_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()                      # clear old gradients
        loss = criterion(model(images), labels)    # forward + loss
        loss.backward()                            # backpropagation (automatic!)
        optimizer.step()                           # update weights
        scheduler.step()                           # adjust learning rate
        running_loss += loss.item() * images.size(0)

    test_acc = evaluate(test_loader)
    print(f"Epoch {epoch:2d}/{EPOCHS} | loss: {running_loss / len(train_ds):.4f}"
          f" | test acc: {test_acc:.2%}")

    if test_acc > best_acc:                        # keep the best model
        best_acc = test_acc
        torch.save(model.state_dict(), "best_model.pt")

print(f"\nBest test accuracy: {best_acc:.2%}")

Using device: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 20.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 475kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.54MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.50MB/s]


Parameters: 468,202
Epoch  1/10 | loss: 0.9955 | test acc: 98.61%
Epoch  2/10 | loss: 0.6510 | test acc: 98.97%
Epoch  3/10 | loss: 0.6171 | test acc: 99.33%
Epoch  4/10 | loss: 0.5974 | test acc: 99.29%
Epoch  5/10 | loss: 0.5848 | test acc: 99.43%
Epoch  6/10 | loss: 0.5746 | test acc: 99.51%
Epoch  7/10 | loss: 0.5679 | test acc: 99.58%
Epoch  8/10 | loss: 0.5630 | test acc: 99.61%
Epoch  9/10 | loss: 0.5585 | test acc: 99.60%
Epoch 10/10 | loss: 0.5554 | test acc: 99.62%

Best test accuracy: 99.62%


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
!pip install -q gradio